# Session 04 · Part 2 — Memory and files

The agent of Part 1 (same prompt, same Skill) forgets everything when a new thread starts. You give it:

| Kind | Where | Lives |
|---|---|---|
| Short-term memory | messages and scratch files of the thread (checkpointer, `StateBackend`) | one thread |
| Long-term memory | `/memories/AGENTS.md` in a store (`StoreBackend` on SQLite) | every thread, restarts |
| Artifact | `/workspace/brief.md` on disk (`FilesystemBackend`, `output/workspace-part2/`) | until you delete it |

<img src="images/b4_short_long_memory.png" alt="short-term and long-term memory" width="400">

Docs: [Memory](https://docs.langchain.com/oss/python/deepagents/memory) ·
[Backends](https://docs.langchain.com/oss/python/deepagents/backends)

## 0. Setup

In [1]:
import logging
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger(__name__)

# ── Keys and model (.env of this session folder) ─────────────────────────────
load_dotenv()
for name in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    if not os.getenv(name, "").strip():
        raise RuntimeError(f"Missing {name}: copy .env.example to .env in the session folder and fill it in")
model = init_chat_model(os.getenv("MODEL", "openai:gpt-6-luna"))
logger.info("Model: %s", model.model_name)

OUT = Path("output")  # everything the agent writes goes here (git-ignored)
OUT.mkdir(exist_ok=True)

INFO | Model: gpt-6-luna


In [2]:
from utils import SKILLS_DIR, ask, context_window, internet_search, routes, system_prompt  # notebooks/utils.py

WINDOW = context_window(model)  # max input tokens, from the model's LangChain profile
logger.info("Context window: %s tokens", f"{WINDOW:,}")

INFO | Context window: 1,050,000 tokens


## 1. A filesystem with three backends
The agent always sees one filesystem. A `CompositeBackend` routes each path prefix to a backend:

- `/` (default) → `StateBackend`: files in the thread state, gone in a new thread.
- `/memories/` → `StoreBackend`: a LangGraph store, shared by all threads. Here a `SqliteStore` file, so it survives restarts.
  The `namespace` says whose data it is; Part 4 makes it per user.
- `/skills/` → `FilesystemBackend`: the Skill folder of Part 1, read from disk.
- `/workspace/` → `FilesystemBackend(virtual_mode=True)`: real files under `output/workspace-part2/`; `..`, `~` and absolute paths are refused.

<img src="images/b4_backends.png" alt="backends" width="400">

In [3]:
import sqlite3

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend, StoreBackend
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.sqlite import SqliteStore

MEMORY_DB = OUT / "memory.sqlite"
WORKSPACE = OUT / "workspace-part2"
WORKSPACE.mkdir(exist_ok=True)


def open_store() -> SqliteStore:
    store = SqliteStore(sqlite3.connect(MEMORY_DB, check_same_thread=False, isolation_level=None))
    store.setup()
    return store


if "store" in globals():
    store.conn.close()  # re-running this cell: close the old connection before deleting the file
MEMORY_DB.unlink(missing_ok=True)  # start clean so the run below is reproducible
store = open_store()
MEMORY_NS = ("research-agent", "memories")  # one agent, one user for now (Part 4: one namespace per user)

# path the agent sees -> (where the files are, what the agent keeps there)
LAYOUT = {
    "/memories/": (StoreBackend(namespace=lambda rt: MEMORY_NS),
                   "AGENTS.md holds the user's lasting preferences; update it when they state one"),
    "/skills/": (FilesystemBackend(root_dir=SKILLS_DIR.resolve(), virtual_mode=True), None),
    "/workspace/": (FilesystemBackend(root_dir=WORKSPACE.resolve(), virtual_mode=True), "deliverables the user will open"),
}
backend = CompositeBackend(default=StateBackend(), routes=routes(LAYOUT))

## 2. Memory as a file that is always loaded
`memory=["/memories/AGENTS.md"]` makes `MemoryMiddleware` put the file into the system prompt inside `<agent_memory>`,
together with guidelines on when to update it with `edit_file`. The file does not exist yet: the agent creates it.
Memory is read **once, when a thread starts**; a change shows up in the next thread.
The system prompt is the one of Part 1; the `<files>` section built from `LAYOUT` tells the agent what `/memories/` is for
(Deep Agents docs: "Document long-term memory structure").

In [4]:
print(system_prompt(LAYOUT)[-320:])  # the shared prompt of Part 1 + a <files> section from LAYOUT

agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt=system_prompt(LAYOUT),
    skills=["/skills/"],
    memory=["/memories/AGENTS.md"],
    backend=backend,
    store=store,
    checkpointer=InMemorySaver(),  # short-term memory: the messages of each thread
)

def chat(question: str, thread_id: str):
    """Stream one turn, then print one context row per model call."""
    return ask(agent, question, thread_id, window=WINDOW)

instructions>

<output_description>
Answer quick questions in 1 to 3 sentences. Put each source url in brackets after the sentence it supports.
</output_description>

<files>
- /memories/ AGENTS.md holds the user's lasting preferences; update it when they state one
- /workspace/ deliverables the user will open
</files>


## 3. Thread `t1`: the user states a preference

In [5]:
result, log = chat("I prepare material for the Berlin energy team. Always give costs in euros and use metric units. "
                  "Please remember that.", "t1")
item = store.get(MEMORY_NS, "/AGENTS.md")
print("\nstored in the store:", item.key if item else None)
print(item.value["content"] if item else "nothing saved")

→ ls({"path": "/memories"})
← ls success: No files found
→ write_file({"file_path": "/memories/AGENTS.md", "content": "For material prepared for the Berlin energy team, always expr…)
← write_file success: Updated file /memories/AGENTS.md
I’ll use euros for costs and metric units in material for the Berlin energy team.
call messages   tokens  in this call
   1        1    3,802  skills index, memory
   2        3    3,855  skills index, memory
   3        5    3,949  skills index, memory

stored in the store: /AGENTS.md
For material prepared for the Berlin energy team, always express costs in euros and use metric units.


### Short-term memory: same thread
The checkpointer keeps the messages of `t1`, so a follow-up question in `t1` sees the earlier turn.

In [6]:
result, log = chat("What did I just ask you to remember?", "t1")
print("messages in thread t1:", len(agent.get_state({"configurable": {"thread_id": "t1"}}).values["messages"]))

You asked me to always give costs in euros and use metric units for material prepared for the Berlin energy team.
call messages   tokens  in this call
   1        7    3,985  skills index, memory
messages in thread t1: 8


## 4. Thread `t2`: a new conversation still knows the preference
No message of `t1` is in `t2`. The preference comes from `/memories/AGENTS.md`, now inside the system prompt.
The brief is an **artifact**: a file outside the context that the user can open; the context keeps only its path.

In [7]:
result, log = chat("Write a brief on the installation costs of heat pumps for apartment buildings.", "t2")
log.show(0)  # first call of t2: the memory block is already in the system prompt
print("\nmemory in the system prompt of t2:", log.system_parts(log.system)["memory: AGENTS.md content"])
print("\nmessages in thread t2:", len(agent.get_state({"configurable": {"thread_id": "t2"}}).values["messages"]))

→ read_file({"file_path": "/skills/research-brief/SKILL.md", "offset": 0, "limit": 1000})
← read_file success: @@ lines 1-15 of 15 @@
---
name: research-brief
description: Write a one-page research brief with cited web so…
→ read_file({"file_path": "/skills/research-brief/references/brief_template.md", "offset": 0, "limit": 1000})
← read_file success: @@ lines 1-15 of 15 @@
# <Topic>: research brief

**Audience:** <who will read it>

## Question
One sentence: …
→ internet_search({"query": "site:co2online.de Wärmepumpe Mehrfamilienhaus Kosten Einbau Kosten Deutschland", "max_results": 5})
→ internet_search({"query": "site:energiewechsel.de Wärmepumpe Mehrfamilienhaus Kosten Förderung Kostencheck", "max_results": 5})
→ internet_search({"query": "site:dena.de Mehrfamilienhaus Wärmepumpe Investitionskosten Studie", "max_results": 5})


INFO | internet_search('site:energiewechsel.de Wärmepumpe Mehrfamilienhaus Kosten Förderung K…) -> ok 3392 ms


← internet_search success: [{"title": "BMWE | Antworten auf häufig gestellte Fragen zur Wärmepumpe (FAQ)", "url": "https://www.energiewec…


INFO | internet_search('site:co2online.de Wärmepumpe Mehrfamilienhaus Kosten Einbau Kosten De…) -> ok 4305 ms


← internet_search success: [{"title": "Luft-Wasser-Wärmepumpe: Kosten, Förderung, Vorteile/Nachteile | co2online", "url": "https://www.co…


INFO | internet_search('site:dena.de Mehrfamilienhaus Wärmepumpe Investitionskosten Studie') -> ok 4624 ms


← internet_search success: [{"title": "Hochtemperatur-Wärmepumpen Einsatz und CO", "url": "https://www.dena.de/fileadmin/dena/Publikation…
→ write_file({"file_path": "/workspace/brief.md", "content": "# Heat-pump installation costs for apartment buildings: resea…)
← write_file success: Updated file /workspace/brief.md
The available sources do not provide a reliable, general installation-cost range for apartment buildings in Germany. [https://www.co2online.de/modernisieren-und-bauen/waermepumpe/luft-wasser-waermepumpe-kosten-foerderung]  
The federal FAQ’s €22,500-per-dwelling figure is an eligible-cost ceiling, not an installation-cost estimate. [https://www.energiewechsel.de/KAENEF/Redaktion/DE/FAQ/FAQ-Uebersicht/BEG/Downloads/beg-faq-241220.pdf?__blob=publicationFile&v=4]  
Brief: `/workspace/brief.md`
call messages   tokens  in this call
   1        1    3,816  skills index, memory
   2        3    4,148  skills index, memory
   3        5    4,283  skills index, memory
   4        

In [8]:
print((WORKSPACE / "brief.md").read_text())

# Heat-pump installation costs for apartment buildings: research brief

**Audience:** City energy team (German context)

## Question
What reliable public evidence is available on the upfront installation cost of heat pumps in apartment buildings in Germany?

## Key findings
- The available co2online cost page distinguishes a 70 m² apartment in a multi-family building for *annual operating costs* (€680 in 2024; €715 forecast for 2025), but its cited installation-cost example is for a single-family home—not a whole apartment building. Those operating-cost figures should not be presented as installation costs. [https://www.co2online.de/modernisieren-und-bauen/waermepumpe/luft-wasser-waermepumpe-kosten-foerderung]
- A federal BEG FAQ search result says that costs up to €22,500 per dwelling are eligible for support in the multi-family-building context described there. This is a subsidy-eligibility ceiling, not an estimate of the building’s installation bill or the amount of subsidy received

## 5. Where each file lives
- `/memories/AGENTS.md` → row in `output/memory.sqlite`
- `/workspace/brief.md` → `output/workspace-part2/brief.md`
- anything else the agent wrote → the state of its thread only

In [9]:
for tid in ("t1", "t2"):
    files = agent.get_state({"configurable": {"thread_id": tid}}).values.get("files", {})
    print(f"thread {tid} state files:", sorted(files) or "none")
print("store:", [(i.namespace, i.key) for i in store.search(MEMORY_NS)])
print("disk: ", sorted(p.name for p in WORKSPACE.iterdir()))

thread t1 state files: none
thread t2 state files: none
store: [(('research-agent', 'memories'), '/AGENTS.md')]
disk:  ['brief.md']


## 6. Restart: long-term memory is still there
A new `SqliteStore` on the same file, as after a kernel restart.

In [10]:
reopened = open_store()
print(reopened.get(MEMORY_NS, "/AGENTS.md").value["content"])
reopened.conn.close()

For material prepared for the Berlin energy team, always express costs in euros and use metric units.


### Try it

In [11]:
question = "What do you remember about me?"  # a new thread: the answer comes from /memories/AGENTS.md
# result, log = chat(question, "t9")

### Done when
- `t1` wrote `/memories/AGENTS.md`; `t2` (installation costs) gave costs in euros without being told again, and its system prompt shows the memory.
- `output/workspace-part2/brief.md` follows the Skill's template; the reopened store still returns the memory file.